# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its `@id`.


In [ ]:
# List available RecordSets, their @id, and main Fields (columns)
all_recordsets = dataset.metadata.record_sets
print(f"Number of RecordSets: {len(all_recordsets)}\n")
for rset in all_recordsets:
    print(f"RecordSet name: {rset.name}, @id: {rset.id}")
    print("  Fields (columns):")
    for field in rset.fields:
        print(f"    - name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis. Use the RecordSet and field `@id`s and `name`s from the overview above.


In [ ]:
# For this dataset, find RecordSets and load them by their @id
record_sets = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")
    print(f"Columns: {df.columns.tolist()}\n")

# For exploration, select the first RecordSet
selected_record_set_id = record_sets[0] if record_sets else None
if selected_record_set_id:
    print(f"Showing first five records from RecordSet: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Choose a numeric field for filtering/normalization.
# List fields in the selected RecordSet for reference
if selected_record_set_id and not dataframes[selected_record_set_id].empty:
    df = dataframes[selected_record_set_id]
    # Guess a numeric field by data type
    recordset_meta = next(r for r in dataset.metadata.record_sets if r.id == selected_record_set_id)
    numeric_field_id = None
    for field in recordset_meta.fields:
        if field.data_type in ['Integer', 'Float', 'Number', 'schema:Integer', 'schema:Float', 'schema:Number']:
            numeric_field_id = field.id
            numeric_field_name = field.name
            break
    if numeric_field_id is not None and numeric_field_name in df.columns:
        # Try to convert the field to numeric
        df[numeric_field_name] = pd.to_numeric(df[numeric_field_name], errors='coerce')
        threshold = df[numeric_field_name].mean() if not df[numeric_field_name].isnull().all() else 0
        filtered_df = df[df[numeric_field_name] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (field name: {numeric_field_name}):\n")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_name}_normalized"] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()
        print(f"\nNormalized {numeric_field_id} for filtered records (field name: {numeric_field_name}):\n")
        print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

        # Find a non-numeric field to group by, e.g. first string field
        group_field_id = None
        group_field_name = None
        for field in recordset_meta.fields:
            if field.data_type in ['Text', 'schema:Text', 'String'] and field.name != numeric_field_name:
                group_field_id = field.id
                group_field_name = field.name
                break
        if group_field_name and group_field_name in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_name).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (field name: {group_field_name}):\n")
            print(grouped_df.head())
        else:
            print("\nNo suitable group-by field found.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and not dataframes[selected_record_set_id].empty and numeric_field_id is not None and numeric_field_name in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_name].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_name} ({numeric_field_id})")
    plt.xlabel(numeric_field_name)
    plt.ylabel('Count')
    plt.show()

    # If there is a group field, plot mean by group
    if group_field_name and group_field_name in df.columns:
        plt.figure(figsize=(10, 5))
        mean_per_group = df.groupby(group_field_name)[numeric_field_name].mean().sort_values()
        mean_per_group.plot(kind='bar')
        plt.title(f"Mean of {numeric_field_name} grouped by {group_field_name}")
        plt.xlabel(group_field_name)
        plt.ylabel(f"Mean {numeric_field_name}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrates how to access and explore a FAIR-compliant mass spectrometry dataset using the `mlcroissant` library.
Key steps included:
- Loading the Croissant metadata and inferring the dataset's structure from `@id`/@name references.
- Exploring available RecordSets and their fields.
- Extracting and analyzing one RecordSet: filtering, normalization, grouping, and visualizing its records.

Refer to Croissant's schema and documentation for further programmatic exploration or to process additional RecordSets in depth.